# Les files — Exercice 4 : recherche et suppression des doublons (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** recherche dans une file, file auxiliaire, réutiliser ses propres fonctions.

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

---
# MISE EN PLACE

> Dans la suite, on utilisera exclusivement uniquement des objets de la classe File définie dans l'exercice 1.
> (Interdit d'utiliser le type list de python.)

On importe la classe `File` de l'exercice 1 : le fichier `file.py` doit être **dans le même dossier** que ce notebook.

La fonction `file_de` (non demandée) sert seulement à fabriquer vite des files **pour vérifier les réponses** : elle reçoit une liste, mais elle ne fait pas partie des solutions. Une file s'affiche sous la forme `tête < ... < queue` : on **défile à gauche** et on **enfile à droite**.

In [1]:
from file import File


def file_de(elements):
    f = File()
    for el in elements:
        f.enfile(el)
    return f


print(file_de(["a", "b", "c"]))

tête < a < b < c < queue


---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Écrire une fonction `est_dans_la_file(f, val)` qui renvoie `True` si la valeur `val` apparaît au moins une fois dans la file `f`, et renvoie `False` sinon. Après l'appel, la file `f` doit être identique à ce qu'elle était initialement.

#### ✅ RÉSOLUTION

In [2]:
def est_dans_la_file(f, val):
    trouve = False
    for i in range(f.taille()):
        el = f.defile()
        if el == val:
            trouve = True
        f.enfile(el)
    return trouve

*Vérification (non demandée) :*

In [3]:
f = file_de([5, 8, 2, 8])
print(est_dans_la_file(f, 2), est_dans_la_file(f, 7))
print("La file après les appels :", f)

assert est_dans_la_file(f, 2)
assert not est_dans_la_file(f, 7)
assert str(f) == str(file_de([5, 8, 2, 8]))   # file inchangée
assert not est_dans_la_file(File(), 1)

True False
La file après les appels : tête < 5 < 8 < 2 < 8 < queue


#### 💡 EXPLICATION PÉDAGOGIQUE

- **La technique clé de tout le TD : la rotation.** Avec une file, on ne voit que la **tête**. Pour regarder tous les éléments, on fait faire **un tour complet** à la file : on défile un élément, on le traite, puis on le **ré-enfile**. Après exactement `f.taille()` rotations, chaque élément est revenu à sa place : la file est **identique** à ce qu'elle était au départ.
  ```python
  for i in range(f.taille()):
      el = f.defile()
      ...            # traiter el
      f.enfile(el)
  ```
- On calcule `f.taille()` **une seule fois**, au départ (c'est ce que fait `range`). Une boucle `while not f.est_vide()` ne s'arrêterait jamais, puisqu'on ré-enfile chaque élément.

- Le booléen `trouve` passe à `True` dès qu'on rencontre `val`, mais **on termine le tour**.
- ⚠️ **Le piège de cet exercice :** avec une liste, on écrirait `return True` dès qu'on trouve la valeur. Ici, la fonction s'arrêterait au milieu du tour et les éléments déjà défilés se retrouveraient **derrière** les autres : la file serait modifiée. Démonstration :

In [4]:
def est_dans_la_file_FAUX(f, val):
    for i in range(f.taille()):
        el = f.defile()
        f.enfile(el)
        if el == val:
            return True         # on sort au milieu du tour !
    return False


f = file_de([5, 8, 2, 8])
est_dans_la_file_FAUX(f, 8)
print(f, " <- l'ordre a changé : 5 est passé derrière")

tête < 2 < 8 < 5 < 8 < queue  <- l'ordre a changé : 5 est passé derrière


---
# QUESTION 2

#### 📝 ÉNONCÉ

> 2. Écrire une fonction `supprime_doublons` qui prend en argument une file `f` et la modifie en place de telle sorte qu'elle ne contienne plus de doublons.

#### ✅ RÉSOLUTION

On garde la **première apparition** de chaque valeur, dans l'ordre. On note les valeurs déjà rencontrées dans une file auxiliaire `vues`.

In [5]:
def supprime_doublons(f):
    vues = File()
    for i in range(f.taille()):
        el = f.defile()
        if not est_dans_la_file(vues, el):
            vues.enfile(el)
            f.enfile(el)

*Vérification (non demandée) :*

In [6]:
f = file_de([3, 1, 3, 2, 1, 3])
supprime_doublons(f)
print(f)
assert str(f) == str(file_de([3, 1, 2]))

g = file_de(["a", "a", "a"])
supprime_doublons(g)
assert str(g) == str(file_de(["a"]))
h = File()
supprime_doublons(h)
assert h.est_vide()

tête < 3 < 1 < 2 < queue


#### 💡 EXPLICATION PÉDAGOGIQUE

- **Pour reconnaître un doublon, il faut se souvenir** des valeurs déjà vues. Les listes sont interdites… mais pas les **files** : on utilise une file `vues`.
- Un tour complet de `f` : un élément **déjà vu** n'est **pas ré-enfilé** (il disparaît) ; sinon, on le note dans `vues` et on le ré-enfile dans `f`. On **réutilise** `est_dans_la_file` de la question 1.

| Élément défilé | Déjà vu ? | `vues` | `f` (partie déjà traitée) |
|:---|:---|:---|:---|
| 3 | non | `3` | `3` |
| 1 | non | `3 1` | `3 1` |
| 3 | **oui** → supprimé | `3 1` | `3 1` |
| 2 | non | `3 1 2` | `3 1 2` |
| 1 | **oui** → supprimé | `3 1 2` | `3 1 2` |
| 3 | **oui** → supprimé | `3 1 2` | `3 1 2` |

- C'est bien **en place** : c'est `f` elle-même qui est modifiée ; `vues` n'est qu'un outil de travail.
- **Coût :** pour chacun des `n` éléments, on parcourt `vues` (jusqu'à `n` éléments) : de l'ordre de `n²` opérations (coût **quadratique**).